In [ ]:
# HW07: Attention, BERTopic and DeepLatent

Remember that these homework work as a completion grade. **You can skip one section of this homework.**

## Attention

In [ ]:
import math
import torch
import tiktoken
from torchtext.vocab import GloVe

text = "Your journey starts with the start"

#TODO Tokenize the sentence with tiktoken (GPT-2)


print("Token IDs:", token_ids)
print("Tokens:", tokens)

In [ ]:
glove = {}
with open (r'glove.6B.50d.txt', 'rb') as f:
  for line in f:
    parts = line.split()
    word = parts[0].decode('utf-8')
    vector = np.array(parts[1:], dtype=np.float32)
    glove[word] = vector

#TODO Convert each token into a GloVe vector. If a token is not found in GloVe, assign a zero vector
embedding_dim = 50
token_vectors = []

#TODO Stack into the input embedding matrix X


In [ ]:
#TODO use the CausalAttention class to generate the context vectors

import torch.nn as nn

class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length,
                dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)            
        self.register_buffer(
           'mask',
           torch.triu(torch.ones(context_length, context_length),
           diagonal=1)
        )            

    def forward(self, x):
        b, num_tokens, d_in = x.shape                   
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2)   
        attn_scores.masked_fill_(                    
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf) 
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        attn_weights = self.dropout(attn_weights)

        context_vec = attn_weights @ values
        return context_vec
    



## BERTopic on Congressional speeches

In [ ]:
# Load a random sample of speeches pronounced in the floor of the US Congress between 1994 and 2024
import pandas as pd
df = pd.read_csv('us_congress_speeches_sample.csv') # Found on the same Github
df['year'] = pd.to_datetime(df['date']).dt.year

print("Number of speeches: {}".format(len(df)))
df.head()

Number of speeches: 28731


,date,text,speaker_bioguide,chamber,party,doc_clean,year
0,2003-06-10,"Mr. ISRAEL. Mr. Speaker, I rise today to com...",I000057,House,Democrat,commend induction successful wrestling coach h...,2003
1,2012-05-08,"Mr. LANGEVIN. Mr. Speaker, Democrats are com...",L000559,House,Democrat,commit reduce deficit balanced way contrast br...,2012
2,2010-03-25,"Mr. BISHOP of Utah. Mr. Speaker, Lori Garver...",B001250,House,Republican,number administrator political give credit pri...,2010
3,1995-01-25,"Mr. KIM. Mr. Chairman, I rise today in suppo...",K000181,House,Republican,balanced budget small business tough run small...,1995
4,2003-05-22,"Mr. BAUCUS. Mr. President, sometime in the n...",B000243,Senate,Democrat,near future eye beholder go conference tax cut...,2003


In [ ]:
#!pip install bertopic[visualization]
from bertopic import BERTopic

#TODO fit a topic model (BERTopic) restricting to 10 topics

In [ ]:
#TODO print out and visualize the top 5 words for each topic. 

# DeepLatent

Now we will estimate a topic model but accounting for covariates (metadata)

In [ ]:
#!pip install deeplatent

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from deeplatent.corpus import Corpus
from deeplatent.models import GTM

vectorizer = CountVectorizer(
    ngram_range=(1, 1),
    max_df=0.95,
    min_df=0.001,
    stop_words="english",
    token_pattern=r"(?u)\b\w{3,}\b"  # only words with 3 or more letters
)

vectorizer.fit(docs)  

print("Number of words in the vocabulary: {}".format(len(vectorizer.get_feature_names_out())))

# 2. Define modalities using the external vectorizer 
modalities = {
    "text": {
        "column": "snippet",
        "views": {
            "bow": {
                "type": "bow",
                "vectorizer": vectorizer
            }
        }
    }
}

#TODO build the train dataset to estimate a GTM using as covariates the party, year and chamber for both prevalence and content. 

train_dataset = Corpus(
    df=df,
    modalities=modalities,
    prevalence="", #TODO add prevalence covs
    content="" #TODO add content covs
)

#TODO provide a brief intuition on why prevalence might matter by year and content might matter by party


In [ ]:
#TODO estimate a GTM (same specs as in the lab). Retrain to the amount of steps that you think necessary for the topics to make sense. Use 10 topics. 


#TODO Did the interpretability increased with respect to BERTopic?

In [ ]:
#TODO feed the topics (first 10 words) into the LLM of your choice (either using and API or manually) and ask it to interpret them

In [ ]:
#TODO Finally, do Republicans use different words by topic? Show the 10 most used words by topic for Republicans